# Day 4

## Tokenizing with code

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [2]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [4]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [11]:
!cd /content/sample_data && git clone https://github.com/hdmeyer/llm_engineering.git

Cloning into 'llm_engineering'...
remote: Enumerating objects: 50128, done.
remote: Counting objects: 100% (21844/21844), done.
remote: Compressing objects: 100% (793/793), done.
remote: Total 50128 (delta 21393), reused 21055 (delta 21050), pack-reused 28284 (from 1)
Receiving objects: 100% (50128/50128), 578.38 MiB | 23.59 MiB/s, done.
Resolving deltas: 100% (29129/29129), done.
Updating files: 100% (5820/5820), done.


In [12]:
%run sample_data/llm_engineering/week1/set_ollama.ipynb

fatal: destination path 'llm_engineering' already exists and is not an empty directory.
/content/sample_data/llm_engineering
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

The 'ollama serve' process is running in the background.
['/content/sample_data/llm_engineering/week1', '/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13']


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [ ]:
from openai import OpenAI

openai = OpenAI()

### A message to OpenAI is a list of dicts

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [32]:
!pkill ollama

In [33]:
import subprocess

# Start the command in a background process
process = subprocess.Popen(["ollama", "serve"])

# The kernel can continue execution while the process runs in the background
print("The 'ollama serve' process is running in the background.")

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

The 'ollama serve' process is running in the background.


In [34]:
#response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response = await ollama_client.chat.completions.create(model="llama3.2:1b", messages=messages)

response.choices[0].message.content

"Hi Ed! How's it going? Is there anything I can help you with or would you like to chat?"

### OK let's now ask a follow-up question

In [35]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [36]:
response = await ollama_client.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"This conversation has just started, so I don't have any information about your name. I'm a text-based AI assistant, and I don't retain any information about individual users. I'm here to help with any questions or topics you'd like to discuss, though!"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [38]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [39]:
response = await ollama_client.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

'Your name on my end is Ed. Is there anything I can help you with or would you like to chat about something in particular?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

